# Assignment 3 - free invoice extraction comparison

Choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. No API key, paid API or code editing is required. Do not purchase compute units.

This runs the non-LLM baseline, Qwen2.5-1.5B, Qwen2.5-7B (4-bit), then a refined hybrid on all 50 PDFs. The larger model is a higher-compute comparison, not a paid API tier: both have zero API fees. If a higher-priced hosted API is mandatory, that is a disclosed substitution rather than full compliance with that interpretation.

Free GPU quota/availability can interrupt a long run. Each invoice is saved immediately. If a runtime remains active, running again resumes its saved responses. At the end, the results ZIP downloads automatically. Also save **File > Download > Download .ipynb** and return both files. Neither model has been run for this new assignment yet; do not present this template as execution evidence.

Sources: [Qwen 1.5B](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Qwen 7B](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct), [4-bit loading](https://huggingface.co/docs/transformers/quantization/bitsandbytes).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pypdf>=5,<7',
    'transformers==4.57.1', 'accelerate>=1,<2', 'bitsandbytes>=0.48,<1', 'requests'], check=True)
print('Dependencies ready. If Colab requests a restart, restart then continue with the next cell.')


In [ ]:
from pathlib import Path
import io, json, zipfile, requests, os, hashlib, datetime, importlib.metadata, shutil
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU. Do not buy compute.')
WORK=Path('/content/assignment3_invoice_run');WORK.mkdir(exist_ok=True)
url='https://codeload.github.com/MVEST22/invoice-extraction-assignment/zip/refs/heads/main'
r=requests.get(url,timeout=120);r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    for entry in z.infolist():
        p=Path(entry.filename)
        if p.is_absolute() or '..' in p.parts:raise ValueError('Unexpected source archive path')
    z.extractall(WORK)
ROOT=WORK/'invoice-extraction-assignment-main'
os.chdir(ROOT)
subprocess.run([sys.executable,'scripts/setup_data.py'],check=True)
subprocess.run([sys.executable,'scripts/preflight.py'],check=True)
print('GPU:',torch.cuda.get_device_name(0))
print('Ground truth is reserved for evaluation and is not passed into either model.')


## Method 1 - non-LLM baseline

In [ ]:
subprocess.run([sys.executable,'methods/method1_programmatic.py'],check=True)
subprocess.run([sys.executable,'evaluation/compare_results.py'],check=True)


## Method 2 - smaller local LLM
The cell prints progress for each of 50 invoices. A malformed model answer is recorded as a failure, not silently replaced with a correct answer.

In [ ]:
subprocess.run([sys.executable,'methods/method2_low_cost.py'],check=True)


## Method 3 - larger local LLM
The smaller model process has exited, freeing GPU memory. This cell loads the larger model with 4-bit weights.

In [ ]:
subprocess.run([sys.executable,'methods/method3_expensive.py'],check=True)


## Method 4 and evaluation
Review Methods 1-3 first. Then run the refined source-text rules, preserve printed totals, and report arithmetic discrepancies separately. Ground truth is never used to fill extraction fields.

In [ ]:
subprocess.run([sys.executable,'evaluation/compare_results.py'],check=True)
shutil.copyfile(ROOT/'evaluation/summary.json',ROOT/'evaluation/initial_three_methods.json')
for script in ['methods/method4_refined.py','evaluation/compare_results.py',
               'scripts/calculate_costs.py','scripts/build_db.py','scripts/build_report.py']:
    subprocess.run([sys.executable,script],check=True)
print((ROOT/'REPORT.md').read_text())


## Download results
Run this last cell even if an earlier cell failed, to preserve partial evidence for troubleshooting. A partial ZIP is clearly marked incomplete.

In [ ]:
from google.colab import files
import csv
DEST=WORK/'results_package';DEST.mkdir(exist_ok=True)
for directory in ['outputs','evaluation','methods','scripts']:
    shutil.copytree(ROOT/directory,DEST/directory,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
for name in ['REPORT.md','README.md','requirements.txt','run_pipeline.py','tests_local.py',
             'invoice_results.db','Assignment_3_Free_Colab.ipynb']:
    if (ROOT/name).exists():shutil.copyfile(ROOT/name,DEST/name)
status={}
for method in ['method1_programmatic','method2_low_cost','method3_expensive','method4_refined']:
    path=ROOT/'outputs'/(method+'.csv')
    if path.exists():
        with path.open(encoding='utf-8-sig',newline='') as f:status[method]=len(list(csv.DictReader(f)))
    else:status[method]=0
manifest={'packaged_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'rows':status,'complete':all(n==50 for n in status.values()),'api_calls':0,'api_spend_usd':0,
    'model_comparison':'Smaller and larger open models on free compute; not purchased API price tiers.',
    'gpu':torch.cuda.get_device_name(0),
    'packages':{p:importlib.metadata.version(p) for p in ['torch','transformers','pypdf','bitsandbytes','accelerate']},
    'input_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((ROOT/'invoices').glob('*.pdf'))},
    'code_sha256':{str(p.relative_to(ROOT)):hashlib.sha256(p.read_bytes()).hexdigest() for d in ['methods','scripts','evaluation'] for p in (ROOT/d).glob('*.py')}}
(DEST/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
(DEST/'EXECUTION_NOTE.txt').write_text('Download your EXECUTED Colab notebook separately: File > Download > Download .ipynb. The included notebook is an unexecuted source template, not execution evidence. Return this ZIP and the executed notebook. See manifest complete/row counts before presenting results as complete.')
archive=shutil.make_archive(str(WORK/'assignment3_results'),'zip',DEST)
print('Rows per method:',status,'Complete:',manifest['complete'])
files.download(archive)
